# NB5 · TD 线性近似：手写半梯度 TD(0)，再做特征设计消融

**对应讲**：L8（值函数近似：从表格到 DQN） · **难度**：level 3 · **预计用时**：约 45 分钟

第八讲把状态价值从「一张表」升级成「一个参数化函数」$\hat v(s, w) = w^\top \phi(s)$。这本笔记本把它跑起来，主线三步：

1. **手写半梯度 TD(0)**：numpy 从零实现 $w \leftarrow w + \alpha\,\delta\,\phi(s)$；
2. **特征设计消融**：同一套训练器，换三套特征（one-hot / 块聚合 / 坐标），各跑 5 个种子；
3. **与表格真值对比**：先解出 $v_\pi$ 作 ground truth，再量每种特征能逼近到哪里。

**前置**：主站 L8 qa 节的推导链「半梯度 TD 目标推导」——尤其"目标侧冻结"那一步；NB0 的 4×4 GridWorld（本本已内嵌同款，无需先跑）。

一句话定位：**这是 DQN 去掉神经网络、只留线性核心的最小版本**——$w$ 就是缩小版的 $\theta$，特征 $\phi$ 就是手工版的"网络表示"。


## 怎么用这本笔记本

- 选中格子按 **Shift + Enter** 运行并跳到下一格；两个 `TODO` 格需要你亲手补完，旁边的 `✅ 自检` 格跑绿才算通过；
- 带宽提示：本本要画图，用到 **matplotlib**——比 NB0 多下载约 **8–10 MB**，之后浏览器强缓存；
- 已知限制：刷新浏览器会清空 kernel 状态，全部重跑即可；改动默认不落盘，想保留请用菜单下载备份；
- 依赖只有 numpy + matplotlib；4×4 GridWorld 与 NB0 同款、已内嵌（自包含单文件）；
- 末尾的 `🏔 挑战` 没有标准答案——那是你自己的实验。


In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

print("Python     :", sys.version.split()[0])
print("numpy      :", np.__version__)
print("matplotlib :", plt.matplotlib.__version__)

assert int(np.__version__.split(".")[0]) in (1, 2), "numpy 主版本异常"
print("就绪：numpy + matplotlib（本本的唯一依赖）")


## 1 · 半梯度 TD(0)：公式与「目标侧冻结」

L8 推导链的结论一行版——预测侧 $\hat v(s,w) = w^\top\phi(s)$，真值未知用自举目标顶替，得到 **TD-Linear**：

$$\delta = \underbrace{r + \gamma\, w^\top\phi(s')}_{\text{目标：含 } w \text{ 却被当作常数}} - w^\top\phi(s)
\qquad\qquad
w \leftarrow w + \alpha\, \delta\, \phi(s)$$

**"半"在哪**：目标 $r + \gamma\, w^\top\phi(s')$ 本身就是 $w$ 的函数，按微积分的规矩，完整梯度应把这一项的链式贡献也算进来；半梯度偏不——**目标侧冻结，只对预测侧求导**。线性情形 $\nabla_w \hat v(s, w) = \phi(s)$，所以更新式里乘的就是 $\phi(s)$ 本身。

报偿与代价（L8 原文）：更新保持 SGD 形状、自举目标只带一步噪声；但更新方向**不再是任何固定目标函数的梯度**——若把目标侧也解冻，靶子跟着 $w$ 跑，"梯度下降保证下山"的直觉就此失效。DQN 的目标网络冻结的正是同一处。


### 1.1 先手算两个 δ

训练开始前 $w = 0$（于是 $\hat v \equiv 0$），在 4×4 网格上手算：

- **s11 右移进 s12（目标）**：$r = +1$，剧集终止、目标不自举 → $\delta = 1 - 0 = +1$；
- **s7 右移撞 s8（禁区弹回）**：$r = -1$，留在 s7 → $\delta = -1 + 0.9 \times 0 - 0 = -1$。

再换一个假想的 $w$（所有状态估值 0.5）重算第二条，看 $\delta$ 怎么变。


In [ ]:
GAMMA = 0.9   # 主站 A4 作业世界的折扣因子（§2 还会对齐一遍规格）

# w = 0：训练开始前
delta_a = 1.0 - 0.0                    # s11 右移进 s12：终止，目标 = r = +1
delta_b = -1.0 + GAMMA * 0.0 - 0.0     # s7 右撞 s8 弹回：r = -1，留在 s7
assert delta_a == 1.0 and delta_b == -1.0

# 假想 v̂ ≡ 0.5（one-hot 下把 w 全填 0.5 的效果）
v_hat = 0.5
delta_c = -1.0 + GAMMA * v_hat - v_hat
print(f"w = 0    时  δ(s7→弹回) = {delta_b:+.1f}")
print(f"v̂ ≡ 0.5  时  δ(s7→弹回) = {delta_c:+.2f}")
assert abs(delta_c - (-1.05)) < 1e-12
print("✅ 手算对上：δ 的符号永远指向『缩小这个差』的方向")


### 1.2 从 TD-Linear 到 DQN：一张对照表

| TD-Linear（本本） | DQN（L8 尾声） |
|---|---|
| 权重 $w \in \mathbb{R}^d$，$d \in \{2, 4, 16\}$ | 网络参数 $\theta$，百万维 |
| 手工特征 $\phi(s)$ | 网络自动学出的表示 |
| $\nabla_w \hat v = \phi(s)$（一行更新） | 反向传播算梯度 |
| 目标侧冻结 $w$ | 目标网络 $\theta_T$（定期同步才解冻） |
| （本本无） | 经验回放，打散样本相关性 |

把网络拿掉、只留线性核，才能看清"半梯度"本身的行为——而**特征设计才是本本真正的主角**。


## 2 · 环境：4×4 GridWorld（自包含）

与 NB0 / 主站 A4 作业世界同一套规格（已内嵌，无需先跑 NB0）：

| 项目 | 值 |
|---|---|
| 网格 | **4×4**，状态编号 s1–s16 |
| 起点 | **s1**（左上角） |
| 禁区 | **s8、s10**（撞上弹回原地、扣分） |
| 目标 | **s12**（进入得 +1，剧集终止） |
| 奖励 | 边界 **−1** / 禁区 **−1** / 目标 **+1** / 其他 **0** |
| 折扣 | **γ = 0.9** |

语义要点（作业代码规则）：分支优先级 **出界 > 目标 > 禁区 > 普通**；编号 **s_i ↔ ((i−1) % 4, (i−1) // 4)**，y 向下增长。


In [ ]:
SIZE = 4
NUM_STATES = SIZE * SIZE          # s1 .. s16
START, TARGET = 1, 12
FORBIDDEN = {8, 10}
REWARDS = {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}

# 与主站 data.js 的 A4 作业配置逐项对齐
assert (SIZE, NUM_STATES, START, TARGET) == (4, 16, 1, 12)
assert FORBIDDEN == {8, 10}
assert REWARDS == {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}
assert GAMMA == 0.9
print("规格就位：4×4 / 起点 s1 / 禁区 s8,s10 / 目标 s12 / γ=0.9")


In [ ]:
ACTION_SPACE = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]   # 下 右 上 左 原（作业代码列序）
DOWN, RIGHT, UP, LEFT, STAY = ACTION_SPACE


def s2xy(s):
    """状态编号 → (x, y)，y 向下增长：s1=(0,0) s8=(3,1) s10=(1,2) s12=(3,2)"""
    i = int(s) - 1
    return i % SIZE, i // SIZE


def xy2s(x, y):
    """(x, y) → 状态编号"""
    return int(y) * SIZE + int(x) + 1


assert len(set(ACTION_SPACE)) == 5
assert s2xy(1) == (0, 0) and s2xy(8) == (3, 1) and s2xy(10) == (1, 2) and s2xy(12) == (3, 2)
assert all(xy2s(*s2xy(s)) == s for s in range(1, NUM_STATES + 1))
print("动作空间与坐标换算就绪：s8=(3,1)  s10=(1,2)  s12=(3,2)")


In [ ]:
class GridWorld:
    """4×4 网格世界：NB0 同款的 numpy 瘦身复刻（自包含单文件版）。

    语义（作业代码规则）：
      出界    → 原地不动，reward = boundary  = -1
      进目标  → 走进目标，reward = target   = +1，done=True
      撞禁区  → 原地弹回，reward = forbidden = -1
      普通/原 → 正常移动，reward = other    =  0
    """

    def __init__(self, size=SIZE, start=START, target=TARGET, forbidden=FORBIDDEN):
        self.size = size
        self.num_states = size * size
        self.start_state, self.target_state = start, target
        self.forbidden_states = set(forbidden)
        self.action_space = ACTION_SPACE
        self.agent_state = start

    def reset(self):
        self.agent_state = self.start_state
        return self.agent_state

    def _get_next_state_and_reward(self, state, action):
        x, y = s2xy(state)
        nxt = np.array([x, y]) + np.array(action)          # numpy 实现转移
        if not (0 <= nxt[0] < self.size and 0 <= nxt[1] < self.size):
            next_state, reward = state, REWARDS["boundary"]      # 1) 出界：原地
        elif xy2s(nxt[0], nxt[1]) == self.target_state:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["target"]   # 2) 目标
        elif xy2s(nxt[0], nxt[1]) in self.forbidden_states:
            next_state, reward = state, REWARDS["forbidden"]     # 3) 禁区：弹回
        else:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["other"]    # 4) 普通
        return next_state, reward

    def step(self, action):
        assert any(tuple(action) == a for a in self.action_space), "非法动作"
        next_state, reward = self._get_next_state_and_reward(self.agent_state, action)
        done = next_state == self.target_state
        self.agent_state = next_state
        return next_state, reward, done, {}


env = GridWorld()
assert env.reset() == START == 1
assert env.num_states == 16 and len(env.action_space) == 5
print("环境就绪：reset() → s1")


In [ ]:
T = np.zeros((NUM_STATES, 5, NUM_STATES))   # T[s-1, a, s'-1]：确定性转移的 one-hot
R = np.zeros((NUM_STATES, 5))               # R[s-1, a]：即时奖励
for s in range(1, NUM_STATES + 1):
    for a in range(5):
        ns, r = env._get_next_state_and_reward(s, ACTION_SPACE[a])
        T[s - 1, a, ns - 1] = 1.0
        R[s - 1, a] = r

assert T.shape == (16, 5, 16) and R.shape == (16, 5)
assert np.allclose(T.sum(axis=-1), 1.0)          # one-hot：转移概率合法
assert T[0, 2, 0] == 1.0 and R[0, 2] == -1.0     # s1 上 → s1，-1
assert T[10, 1, 11] == 1.0 and R[10, 1] == 1.0   # s11 右 → s12，+1
assert T[6, 1, 6] == 1.0 and R[6, 1] == -1.0     # s7 右 → 弹回 s7，-1
print("T:", T.shape, " R:", R.shape, "——动力学张量化完成")


## 3 · 固定策略 π 与真值 v_π

本本的重点是**值近似**，不是策略设计——策略不学习，直接取现成的：**value iteration 解出 v\*，取 greedy**。三个约定先行：

- **确定性策略 + 确定性环境** ⇒ δ 没有随机成分，收敛地板只由特征决定——消融实验要的正是这口干净对照（随机版留给末尾挑战 4）；
- **exploring starts（探索起点）**：每个回合从**随机抽取的非终止状态**出发。这不是可选项：one-hot 的"表格等价"要求**每个状态都被反复更新**，只从 s1 出发的话，角落状态永远轮不到；
- **终止约定**：目标 s12 是终止态，$v_\pi(s_{12}) \equiv 0$（进站拿 +1 后不自举）；后面一切误差都只在 15 个非终止状态上度量。

真值 $v_\pi$ 用**策略评估的线性方程组**解出：$v = r_\pi + \gamma P_\pi v$，在 15 个非终止状态上是良定的线性方程组——它是全本 `‖v̂ − v_π‖` 的 ground truth。


In [ ]:
def value_iteration(gamma=GAMMA, tol=1e-13, max_sweeps=1000):
    """值迭代解 v*（终止约定：v(s12) ≡ 0，进目标不自举）。返回 (v*, greedy 动作)。"""
    v = np.zeros(NUM_STATES)
    for _ in range(max_sweeps):
        q = R + gamma * (T @ v)            # q[s,a] = r(s,a) + γ·v(ns)；v[s12-1]=0 自动生效
        v_new = q.max(axis=1)
        v_new[TARGET - 1] = 0.0            # 终止态价值恒 0
        if np.abs(v_new - v).max() < tol:
            return v_new, q.argmax(axis=1)
        v = v_new
    raise RuntimeError("value iteration 未收敛")


V_STAR, A_STAR = value_iteration()

EVAL_MASK = np.arange(NUM_STATES) != TARGET - 1     # 15 个非终止状态的 mask
EVAL_IDX = np.where(EVAL_MASK)[0] + 1               # 对应的状态编号（1..16，无 s12）

assert V_STAR[TARGET - 1] == 0.0
print(f"v* 就绪：非终止状态取值 [{V_STAR[EVAL_MASK].min():.3f}, {V_STAR[EVAL_MASK].max():.3f}]")
print(np.round(V_STAR.reshape(SIZE, SIZE), 3))


In [ ]:
EPS_SOFT = 0.0   # 贪心（确定性）策略。🏔 挑战 4：改成 0.2 重跑消融，看误差带会怎样

PI = np.full((NUM_STATES, 5), EPS_SOFT / 5)                 # π(a|s)：行=状态，列=动作
PI[np.arange(NUM_STATES), A_STAR] += 1.0 - EPS_SOFT         # greedy 动作拿大头

assert np.allclose(PI.sum(axis=1), 1.0, atol=1e-12)         # 每行是合法分布
assert np.allclose(PI[np.arange(NUM_STATES), A_STAR], 1.0 - EPS_SOFT + EPS_SOFT / 5)
print("策略就绪：greedy(v*)" + ("（确定性）" if EPS_SOFT == 0 else f"（ε-软化 ε={EPS_SOFT}）"))


In [ ]:
P_PI = np.einsum("sa,sat->st", PI, T)      # 16×16：π 下的转移矩阵
R_PI = np.einsum("sa,sa->s", PI, R)      # 16  ：π 下的期望即时奖励

# 真值 v_π：解 (I − γ·P_π) v = r_π，其中 v(s12) ≡ 0 —— 只在 15 个非终止状态上解
V_PI = np.zeros(NUM_STATES)
A_eq = (np.eye(NUM_STATES) - GAMMA * P_PI)[np.ix_(EVAL_MASK, EVAL_MASK)]
V_PI[EVAL_MASK] = np.linalg.solve(A_eq, R_PI[EVAL_MASK])

residual = np.abs((R_PI + GAMMA * (P_PI @ V_PI) - V_PI)[EVAL_MASK]).max()
assert residual < 1e-10, "v_π 必须满足 Bellman 方程"
assert np.allclose(V_PI[EVAL_MASK], V_STAR[EVAL_MASK], atol=1e-9)   # 交叉验证：评估 greedy(v*) 恰得 v*
print(f"真值 v_π 就绪（Bellman 残差 {residual:.1e}）—— 评估 greedy(v*) 与 v* 逐状态一致")
print(np.round(V_PI.reshape(SIZE, SIZE), 3))


### 3.1 读一眼 v_π 的地形

- 价值从左上（≈0.66）向目标 (3,2) 一路爬到 1.0；**s9 是低谷**——离目标最远，还隔着 s10 禁区；
- s8/s10 两块禁区把中间压出一条"走廊"，这张地形**不是**一张斜放的平面——2 维坐标特征马上会撞在这上面；
- 这块 4×4 地形就是三套特征的拟合对象：**特征张出多大的函数空间，v̂ 的天花板就有多高**。


## 4 · TODO 1 · 三套特征函数

线性近似里，**特征就是你注入的先验**。三套设计，维度 16 → 4 → 2 递减：

| 特征 | 维度 | 设计思想 | 预期 |
|---|---|---|---|
| `phi_onehot` | 16 | 每个状态独占一维——**表格法的函数近似形态** | 特征完备 ⇒ 投影=真值，逼近上界基准 |
| `phi_agg` | 4 | 4×4 划成四个 **2×2 块**，块内 4 个状态共享一个权重 | 泛化换精度：块内真值不同 ⇒ 偏差消不掉 |
| `phi_coord` | 2 | 归一化坐标 $[x/3,\, y/3]$ | 最强压缩：v̂ 只能是一张**过原点的平面** |

**API 约定**：`featurizer(s)`，`s ∈ 1..16`，返回一维 `np.ndarray`（float）。提示：`s2xy(s)` 给 `(x, y)`（均从 0 数起）；块编号用 `(y // 2) * 2 + (x // 2)`。


In [ ]:
# TODO 1 · 三套特征函数（把三个 raise NotImplementedError 补完）

def phi_onehot(s):
    """16 维 one-hot：φ(s) = e_{s-1}。与表格法一一对应——近似能力的上界基准。"""
    # TODO: 构造 16 维零向量，把第 s-1 位设成 1.0，返回（形状必须是 (16,)）
    raise NotImplementedError("TODO 1: 完成 phi_onehot")


def phi_agg(s):
    """4 维块聚合：把 4×4 网格划成四个 2×2 块，φ(s) = e_{block(s)}。

    块编号 block = (y // 2) * 2 + (x // 2) ∈ {0,1,2,3}——同一块内的 4 个状态共享同一个权重。
    """
    # TODO: 用 s2xy 求 (x, y)，算块编号，返回对应 one-hot（形状必须是 (4,)）
    raise NotImplementedError("TODO 1: 完成 phi_agg")


def phi_coord(s):
    """2 维归一化坐标：φ(s) = [x/3, y/3]。最强压缩——v̂ 只能是一张过原点的平面。"""
    # TODO: 用 s2xy 求 (x, y)，返回 np.array([x / 3, y / 3])（float，形状 (2,)）
    raise NotImplementedError("TODO 1: 完成 phi_coord")


In [ ]:
# ✅ 自检 · 特征函数（三套 API / 形状 / 语义全对才放行）
assert phi_onehot(1).shape == (16,) and phi_onehot(1).sum() == 1.0
assert np.allclose(phi_onehot(16), np.eye(NUM_STATES)[15])     # s16 → e_16

assert phi_agg(1).shape == (4,) and phi_agg(1).sum() == 1.0
assert np.allclose(phi_agg(1), phi_agg(2))                     # s1=(0,0) 与 s2=(1,0) 同属左上块
assert not np.allclose(phi_agg(1), phi_agg(3))                 # s3=(2,0) 属右上块
assert np.allclose(phi_agg(15), phi_agg(16))                   # s15、s16 同属右下块

assert phi_coord(1).shape == (2,) and np.allclose(phi_coord(1), [0.0, 0.0])
assert np.allclose(phi_coord(16), [1.0, 1.0])                  # s16=(3,3) → [1, 1]
assert np.allclose(phi_coord(12), [1.0, 2 / 3])                # s12=(3,2) → [1, 2/3]
print("✅ 三套特征就绪：one-hot 16 维 / 块聚合 4 维 / 坐标 2 维")


## 5 · TODO 2 · 半梯度 TD(0) 训练器

签名固定：`train(featurizer, alpha, episodes, seed) -> (w, errors)`

- `w`：学到的权重（d 维）；`errors[e]`：第 e+1 个回合结束后的 $\max_{s\,\text{非终止}}|\hat v(s,w) - v_\pi(s)|$（收敛曲线就靠它）；
- **随机性只从** `rng = np.random.default_rng(seed)` **来**：回合起点（exploring start）与动作采样两处；
- 半梯度细节再念一遍：**终止**时目标 = $r$（不自举）；**非终止**时目标 = $r + \gamma\, w^\top\phi(s')$——这行里的 $w$ **不参与求导**，更新只乘预测侧的 $\phi(s)$。

实现提示：

1. 开局先把特征矩阵堆出来：`PHI = np.stack([featurizer(s) for s in range(1, NUM_STATES + 1)])`（16×d），步内用 `PHI[s - 1]` 当 $\phi(s)$，别每步重算；
2. 环境一步：`ns, r = env._get_next_state_and_reward(s, ACTION_SPACE[a])`，`done = (ns == TARGET)`；
3. 误差：`np.abs((PHI @ w)[EVAL_MASK] - V_PI[EVAL_MASK]).max()`。


In [ ]:
# 覆盖体检：为什么敢指望每个状态都学到？——6000 个探索起点的分布
rng_probe = np.random.default_rng(0)
starts = np.array([rng_probe.choice(EVAL_IDX) for _ in range(6000)])
counts = np.bincount(starts, minlength=NUM_STATES + 1)[1:]     # counts[i] = 状态 i+1 被起访次数

assert counts[TARGET - 1] == 0                                  # 终止态从不作为起点
assert counts[EVAL_MASK].min() > 300, "每个非终止状态都应被充分起访"
print(f"6000 个起点：非终止状态最少起访 {counts[EVAL_MASK].min()} 次、最多 {counts[EVAL_MASK].max()} 次")
print("覆盖成立——one-hot『表格等价』的前提就位（L8：特征完备 + 充分访问 ⇒ 投影退化为真值）")


In [ ]:
# TODO 2 · 半梯度 TD(0) 训练器（补完下面的 train）

def train(featurizer, alpha, episodes, seed):
    """半梯度 TD(0)：w ← w + α·δ·φ(s)，δ = r + γ·wᵀφ(s′) − wᵀφ(s)（目标侧冻结）。

    返回 (w, errors)：errors[e] = 第 e+1 个回合后的 max |v̂ − v_π|（非终止状态）。
    """
    rng = np.random.default_rng(seed)
    PHI = np.stack([featurizer(s) for s in range(1, NUM_STATES + 1)])   # 16×d
    w = np.zeros(PHI.shape[1])
    errors = np.empty(episodes)

    # TODO: 外层 episode 循环
    #   1) exploring start：s = int(rng.choice(EVAL_IDX))——从非终止状态均匀出发
    #   2) 内层 step 循环（直到 s == TARGET）：
    #      a. 动作采样：a = int(rng.choice(5, p=PI[s - 1]))
    #      b. 环境一步：ns, r = env._get_next_state_and_reward(s, ACTION_SPACE[a])；done = (ns == TARGET)
    #      c. TD 目标（目标侧冻结！）：done 时 target = r；否则 target = r + GAMMA * (PHI[ns-1] @ w)
    #      d. δ = target − PHI[s-1] @ w
    #      e. 半梯度更新：w = w + alpha * δ * PHI[s-1]（整行 φ(s) 就是 ∇v̂）
    #      f. s = ns
    #   3) 记录：errors[ep] = np.abs((PHI @ w)[EVAL_MASK] - V_PI[EVAL_MASK]).max()
    raise NotImplementedError("TODO 2: 完成 train")
    return w, errors


In [ ]:
ALPHA, EPISODES = 0.2, 6000     # 常步长（确定性环境无随机地板，不需要衰减）

w_onehot, err_onehot = train(phi_onehot, ALPHA, EPISODES, seed=0)

assert err_onehot[0] > 0.5, "开局 w=0，误差应接近 max v_π = 1.0"
assert err_onehot[-1] < 1e-2, "one-hot 特征下 TD(0) 应收敛到表格真值（阈值 1e-2）"
print(f"one-hot：开局误差 {err_onehot[0]:.3f} → 末端误差 {err_onehot[-1]:.2e}（阈值 1e-2）")
print("✅ 表格等价成立：特征完备时，半梯度 TD 的不动点就是 v_π")


In [ ]:
PHI_1h = np.stack([phi_onehot(s) for s in range(1, NUM_STATES + 1)])
v_hat = PHI_1h @ w_onehot

print("v_π（左） vs v̂ one-hot（右），终止态 s12 标记为 T：\n")
for y in range(SIZE):
    row_pi = "  ".join(f"{V_PI[y * SIZE + x]:+.3f}" for x in range(SIZE))
    row_h = "  ".join("  T   " if (y * SIZE + x + 1) == TARGET else f"{v_hat[y * SIZE + x]:+.3f}"
                      for x in range(SIZE))
    print(f"  {row_pi}   |   {row_h}")
print(f"\n非终止状态最大偏差：{np.abs(v_hat[EVAL_MASK] - V_PI[EVAL_MASK]).max():.2e}")


### 5.1 为什么能到机器精度级别

实测末端误差在 $10^{-15}$ 量级——远低于 1e-2 的阈值。这不是巧合，是 L8 的理论在数值上显形：

- **特征完备**（one-hot 张满 $\mathbb{R}^{16}$）⇒ 投影 Bellman 误差的不动点**退化为真值**——表格法是函数近似的特例；
- **充分访问**（exploring starts）保证每个坐标 $w_s$ 都被自己的 Bellman 方程驱动；确定性环境下这 essentially 就是异步值迭代，收敛是 $\gamma$-几何级的；
- 换句话说：`1e-2` 这条线在 one-hot 这里根本不构成约束——**约束来自特征，不来自算法**。下面换特征试试。


## 6 · 消融：三套特征 × 5 个种子

同一套 `train`、同一组超参（α = 0.2，6000 回合），只换 `featurizer`——**特征是唯一变量**；5 个种子换的是探索起点的顺序。跑之前先立预期：

- **one-hot**：一路下探到机器精度（16 个参数各管各的，毫无泛化）；
- **agg**：快速下到 **≈0.17 的地板**——块内 4 个状态真值不同但权重共享，这是**结构性偏差**，不是样本不够；
- **coord**：几乎立刻停在 **≈0.66 的地板**——每步更新都在"平均"所有状态，样本效率极高，但一张过原点的平面装不下这块地形。

看**带**（±1 个标准差）：确定性环境 + 确定性策略下，one-hot 与 coord 的不动点唯一，5 条曲线基本重合；agg 因块内 4 个状态的更新互相拉扯，常步长下留一点种子间差异。浏览器内核较慢，下面这格约需 1–3 分钟。


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
FEATURES = {"one-hot (16d)": phi_onehot, "aggregate (4d)": phi_agg, "coords (2d)": phi_coord}

curves = {}                                    # name -> (5, EPISODES) 误差曲线
for name, feat in FEATURES.items():
    curves[name] = np.stack([train(feat, ALPHA, EPISODES, seed=sd)[1] for sd in SEEDS])

for name, c in curves.items():
    print(f"{name:18s} 5 seeds 末端误差：{np.round(c[:, -1], 4)}")


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
xs = np.arange(EPISODES)
FLOOR = 1e-16          # 机器精度地板：log 轴画不出 0，垫个底让 one-hot 的线可见

for name, c in curves.items():
    mean = np.maximum(c.mean(axis=0), FLOOR)
    std = c.std(axis=0)
    ax.plot(xs, mean, lw=1.6, label=name)
    ax.fill_between(xs, np.maximum(mean - std, FLOOR), mean + std, alpha=0.18)

ax.axhline(1e-2, color="gray", ls="--", lw=1.0, label="assert threshold 1e-2")
ax.set_yscale("log")
ax.set_xlabel("episode")
ax.set_ylabel(r"max |v_hat(s) - v_pi(s)|, non-terminal states")
ax.set_title("Semi-gradient TD(0) under three feature designs")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ✅ 自检 · 末端严格序（TAIL=100：每 seed 最后 100 回合误差均值，再对 seed 平均）
TAIL = 100
end_err = {name: float(c[:, -TAIL:].mean()) for name, c in curves.items()}
e_onehot = end_err["one-hot (16d)"]
e_agg = end_err["aggregate (4d)"]
e_coord = end_err["coords (2d)"]

print(f"末端误差   one-hot = {e_onehot:.2e}   aggregate = {e_agg:.4f}   coords = {e_coord:.4f}")
assert e_onehot < 1e-2, "one-hot 应收敛到表格真值"
assert e_onehot < e_agg < e_coord, "严格序应成立：one-hot < aggregate < coords"
print("✅ 严格序成立：one-hot < 块聚合 < 坐标 —— 特征越省，地板越高")


## 7 · 泛化换精度：这堂课的直觉版

三张曲线放在一起读，三个问题：

1. **coord 为什么最快触底、又停得最高？** 2 个参数意味着每步更新都在向**所有状态**广播信息（x/y 相近的状态共享梯度）——样本效率极高；代价是函数空间只剩一张过原点的平面，而 v_π 的地形有禁区压出的走廊和 s9 低谷，平面装不下 → 偏差地板 ≈0.66。
2. **agg 的 ≈0.17 从哪来？** 块内 4 个状态共享权重，学到的权重是它们真值的某种"平均"——块内离散度就是偏差下界。**结构性偏差**：再多样本也消不掉，只能换特征。
3. **one-hot 为什么能到机器精度？** 16 个参数互不共享，等于把表格摊开写——**零泛化换零偏差**。代价是每一格都要自己的样本喂（幸好 exploring starts 兜底），状态一多就爆炸——正是 L8 开篇"状态空间爆炸的三笔账"。

一句话：**特征数 ↓ = 泛化 ↑ + 精度地板 ↑**——注入的先验越强，前期越快、天花板越低。DQN 的网络学的是"自动特征"：介于这两个极端之间的高维平滑参数化。函数近似的全部艺术，就在这条权衡曲线上选点。


## 8 · 🏔 挑战（无答案，做出来就是你的）

1. **加一维偏置 φ_bias = 1**：把 `phi_coord` 扩成 `[x/3, y/3, 1.0]` 重跑消融——末端误差能降多少？为什么"过原点"的限制解除后平面就能贴住这块地形？再问：给 `phi_onehot` 加偏置维还有意义吗（提示：one-hot 已能表示任意 16 维向量，新维与谁线性相关）？
2. **画出权衡曲线**：以**参数量**为横轴、"误差首次降到 0.7 以下所需回合数"与"末端地板"为双纵轴，把三套特征标上去；再造一套 8 维特征（比如按**行** one-hot），先预测它落在哪，再实测验证。
3. **解冻目标侧**：把更新改成对目标也求导的"全梯度"版本（目标里的 $w^\top\phi(s')$ 也贡献梯度）——它还收敛到同一点吗？观察前几十个回合的曲线形状与半梯度的差别（这正是 L8「追自己 → 目标网络」动机的实验版）。
4. **打破确定性**：把 `EPS_SOFT` 改成 0.2 重跑消融——误差带发生了什么？one-hot 还压得住 1e-2 吗？联系 L6 的步长条件想想：常步长在随机目标下意味着什么，什么时候必须衰减？

（挑战 3、4 要改 `train` / `PI` 本身——改前留个副本，别把跑绿的版本弄丢了。）


## 9 · 回顾

- **手写**：三套特征函数 + 半梯度 TD(0) 训练器（$w \leftarrow w + \alpha\delta\phi(s)$，目标侧冻结）；
- **验证**：one-hot 末端误差 ~$10^{-15}$（表格等价，对 1e-2 阈值富余十三个数量级）；严格序 **one-hot < 块聚合(≈0.17) < 坐标(≈0.66)**；
- **学到的三层**：
  - 算法层——半梯度的"半" = 目标含 $w$ 却当作常数；线性时 $\nabla_w\hat v = \phi(s)$，一行更新；
  - 理论层——TD 不动点是 PBE 的投影：特征完备（one-hot）时退化为真值；
  - 工程层——特征 = 先验：维度越省、泛化越强、精度地板越高；探索覆盖是表格等价的前提。
- 这套 `train(featurizer, ...)` 接口就是 DQN 里"换引擎"的插槽——NB6 把它换成策略梯度，值近似变成策略近似。


## 🎓 下一步

- **NB6 · REINFORCE 与 baseline**（L9/L10）：numpy 手写策略梯度，验证 baseline 只降方差不改期望；
- 想看**真网络**的 DQN：主站 L8 尾声有目标网络与经验回放的完整骨架，NB6 结尾给了 Colab 上传指引；
- 老规矩：刷新浏览器会清空 kernel，跑绿的结果想保留就下载笔记本。

特征决定天花板，算法决定你多快摸到它——带着这条曲线，去见策略梯度。
